# openEO quickstart

This notebook demonstrates a minimal openEO workflow using the Copernicus Data Space backend.

- Basic discovery works anonymously.
- Running processing jobs requires OIDC authentication with a Copernicus Data Space account.
- There is no single global openEO API key to add here.
- The NDVI example builds a process graph first; job execution is opt-in because it can consume credits.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "openeo_demo.py").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from openeo_demo import (
    DEFAULT_BACKEND_URL,
    build_sentinel2_ndvi_cube,
    connect_to_backend,
    describe_collection,
    summarize_backend,
)

DEFAULT_BACKEND_URL

## Connect and inspect the backend

This creates an unauthenticated connection and reads public backend metadata.

In [ ]:
connection = connect_to_backend()
summarize_backend(connection, max_collections=12)

## Inspect Sentinel-2 L2A metadata

In [ ]:
sentinel2 = describe_collection(connection, "SENTINEL2_L2A")
{
    "id": sentinel2["id"],
    "title": sentinel2["title"],
    "bands": sentinel2["bands"][:12],
}

## Optional authentication

Leave this disabled for discovery. Set `AUTHENTICATE = True` when you want to run jobs or access protected endpoints.

In [ ]:
AUTHENTICATE = False

if AUTHENTICATE:
    connection.authenticate_oidc()

## Build an NDVI process graph

The area below is a small bounding box around Malmo, Sweden. The cell builds the graph but does not submit it to the backend.

In [ ]:
spatial_extent = {
    "west": 12.90,
    "south": 55.50,
    "east": 13.10,
    "north": 55.70,
}
temporal_extent = ["2025-06-01", "2025-06-15"]

ndvi_result = build_sentinel2_ndvi_cube(
    connection,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    max_cloud_cover=30,
)

ndvi_result.to_json()

## Optional job execution

Only enable this after authenticating. It submits the NDVI graph as a backend batch job and downloads the result to `outputs/ndvi`.

In [ ]:
RUN_BACKEND_JOB = False

if RUN_BACKEND_JOB:
    job = ndvi_result.create_job(title="Sentinel-2 NDVI quickstart")
    job.start_and_wait()
    job.download_results("outputs/ndvi")